[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GSTT-CSC/summer-school-tutorials/blob/main/day2_qms_regs/QMS_testing_and_validation.ipynb)

# CSC Summer School Day 2 — Testing & Validation

This is the **second half** of the HandOsteo QMS tutorial. In
[`QMS_requirements_and_design.ipynb`](QMS_requirements_and_design.ipynb) you wrote the
requirements, hazard log, and design specification. Here you put them to work:

- **Verification** — *did we build the software right?* Write unit tests that check the
  code against the **System Design Specification (SDS)**.
- **Validation** — *did we build the right software?* Write manual test cases that check
  the system against the clinical need.

You don't need to have finished the first notebook — this one **loads a complete worked
QMS automatically**, so you can focus entirely on testing.

### What HandOsteo does (recap)

It takes an AP hand X-ray (DICOM), validates the modality, classifies the view, measures
the second metacarpal, and returns a report. The stub in `src/handosteo.py` reproduces
that interface without a trained model — and it ships with **two deliberate bugs** for
you to catch.

In [ ]:
# Make sure we have all our dependencies installed in Colab
%pip install -q rdm pandas pytest pyyaml pydicom ipytest pillow dicom2nifti nibabel svgwrite ipython

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# Clone data + helpers if running on a fresh Colab runtime
if not os.path.isdir("day2_data") or not os.path.exists("src/notebook_helpers.py") or not os.path.exists("conftest.py"):
    !git clone --depth=1 -b main https://github.com/GSTT-CSC/summer-school-tutorials.git _repo
    !cp -rn _repo/day2_qms_regs/day2_data .
    !cp -rn _repo/day2_qms_regs/src .
    !cp -r _repo/day2_qms_regs/conftest.py .
    !rm -rf _repo

if "." not in sys.path:
    sys.path.insert(0, ".")

# This notebook is about TESTING — load the complete worked QMS so the requirements and
# design specs are the real HandOsteo ones (not the Eratosthenes placeholders).
for src_file in Path("day2_data/data_solutions").iterdir():
    shutil.copy(src_file, f"day2_data/data/{src_file.name}")
print("Worked HandOsteo QMS data loaded into day2_data/data/")

import ipytest
ipytest.autoconfig()

# The boring boilerplate (YAML printing, DICOM plotting, the diagrams, the behaviour
# table, the unit-test record) lives in src/notebook_helpers.py so the cells below stay
# focused on testing.
from src.notebook_helpers import (
    show_yaml,
    show_design_specs,
    plot_dicom_inputs,
    render_behaviour_table,
    show_handosteo_pipeline,
    show_qms_process_flow,
    show_unit_test_record,
)

### The HandOsteo pipeline at a glance

Four classes run in sequence — each one is a stage you will verify below. The
`DicomLoader` and `ViewClassifier` (the two stages with their own design specs,
SDS-001 and SDS-002) are the focus of this notebook's unit tests.

`DicomLoader` → `ViewClassifier` → `MCPMeasurer` → `ReportGenerator`

In [ ]:
# The HandOsteo pipeline at a glance — four classes, four stages.
show_handosteo_pipeline()

# 5. Verification

Verification asks: **have we built the software according to the design specification?**

This notebook zeroes in on the two design specs that govern HandOsteo's input safety:

| SDS | Title | Design intent |
|---|---|---|
| **SDS-001** | DICOM input validation | Only accept DICOMs whose `Modality` is `CR` or `DX`. |
| **SDS-002** | Image view classifier | Only process `AP`/`PA` views; reject oblique views. |

The cell below prints these design specs straight from the loaded QMS data — these are
the statements your unit tests must verify.

The diagram below shows the full QMS traceability chain. **Verification** is the
unit-testing step — checking the code against the System Design Specification — and is
exactly what you do in this section.

In [ ]:
# Where verification sits in the QMS traceability chain.
show_qms_process_flow()

In [ ]:
# Show the design specs this notebook tests against (SDS-001 and SDS-002).
show_design_specs("day2_data/data/requirements.yml", ids=("SDS-001", "SDS-002"))

### See how the stub behaves today

We provide stub classes in `src/handosteo.py` that mimic the real pipeline. They are
intentionally imperfect: the loader currently **accepts the wrong modality**, and the
view classifier currently **accepts oblique views** — both should be rejected.

We have 7 synthetic hand X-rays in `day2_data/xray/dicom/`:

| File(s) | Modality | View | Should be |
|---|---|---|---|
| `AP_1`, `AP_2`, `AP_3` | CR | AP | ✅ accepted |
| `Ob_1`, `Ob_2` | CR | OBL (oblique) | ❌ rejected (SDS-002) |
| `CT_1` | CT | AP | ❌ rejected (SDS-001) |
| `MR_1` | MR | AP | ❌ rejected (SDS-001) |

Run the next cells to see the inputs, then watch the stub wrongly accept the obliques and
the CT/MR images.

In [ ]:
# Plot the seven synthetic inputs (green border = should be accepted, red = rejected).
plot_dicom_inputs("day2_data/xray/dicom")

In [ ]:
# Set up the HandOsteo pipeline — four stages, each its own class in src/handosteo.py.
import pydicom
from src.handosteo import DicomLoader, ViewClassifier, MCPMeasurer, ReportGenerator

loader = DicomLoader()          # SDS-001: load the DICOM, validate Modality is CR/DX
classifier = ViewClassifier()   # SDS-002: classify the view, reject obliques
measurer = MCPMeasurer()        # measure the second metacarpal → MCP %
report_gen = ReportGenerator()  # turn the measurement into a report

# Now apply the pipeline to every DICOM and record what each stage does. Each stage
# can raise ValueError to reject an input; if it does, we stop and mark it REJECTED.
rows = []
for dcm_path in sorted(Path("day2_data/xray/dicom").glob("*.dcm")):
    raw = pydicom.dcmread(str(dcm_path))
    try:
        ds = loader.load(str(dcm_path))      # SDS-001: should reject non-CR/DX
        view = classifier.classify(ds)       # SDS-002: should reject oblique
        result = measurer.measure(ds)
        report = report_gen.generate(result)
        outcome = report.decode().splitlines()[-1].replace("Classification: ", "")
        rows.append({
            "File": dcm_path.name, "Modality": getattr(raw, "Modality", "?"),
            "View": view, "MCP (%)": f"{result['MCP']:.1f}",
            "HandOsteo says": outcome, "Status": "✓ ACCEPTED",
        })
    except ValueError:
        rows.append({
            "File": dcm_path.name, "Modality": getattr(raw, "Modality", "?"),
            "View": getattr(raw, "ViewPosition", "?"), "MCP (%)": "—",
            "HandOsteo says": "—", "Status": "✗ REJECTED",
        })

# The helper just makes a nice table out of the rows we collected above.
render_behaviour_table(rows)

### Write your verification tests

The cell below contains four tests, each linked to a design spec with
`@pytest.mark.sds("SDS-00X")`.

Every test builds a mock DICOM file with the `make_dataset(modality=..., view=...)` and saves it to disk with `save_mock`

Two tests are already complete and show the pattern. The other two are **incomplete** and pass when they should fail

Your task is to complete the remaining two tests:
- **SDS-001 (`DicomLoader`)** — an unsupported **modality** must be rejected.
- **SDS-002 (`ViewClassifier`)** — an unsupported **view** must be rejected.

Hint: `assert` is a keyword that checks whether a condition is true. If the condition is false, it stops the program by raising an error.

If you're stuck, the solutions can be loaded a couple of cells below

In [ ]:
%%ipytest -qq
# ✏️ YOUR TESTS — complete the two tests marked TODO.
# Every test builds its own mock DICOM with make_dataset(), so you control exactly what
# goes in — no real files needed. The @pytest.mark.sds(...) markers link each test to a
# design spec (SDS-001, SDS-002) and build the unit test record.
import pytest
import pydicom
import pydicom.uid
from pydicom.dataset import FileMetaDataset

from src.handosteo import DicomLoader, ViewClassifier


def make_dataset(modality="CR", view="AP"):
    """Build a minimal in-memory DICOM to use as a mock pipeline input."""
    fm = FileMetaDataset()
    fm.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian
    fm.MediaStorageSOPClassUID = pydicom.uid.SecondaryCaptureImageStorage
    fm.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
    ds = pydicom.FileDataset("", {}, file_meta=fm, preamble=b"\0" * 128)
    ds.Modality = modality
    ds.ViewPosition = view
    return ds


def save_mock(ds, tmp_path):
    """Write a mock dataset to a temp .dcm and return its path (for DicomLoader)."""
    path = tmp_path / "mock.dcm"
    ds.save_as(path)
    return str(path)


class TestHandOsteo:
    # ── SDS-001: DicomLoader accepts supported modalities, rejects the rest ──
    # load() reads from a path, so we save our mock to pytest's tmp_path first.
    @pytest.mark.sds("SDS-001")
    def test_supported_modality_accepted(self, tmp_path):
        path = save_mock(make_dataset(modality="CR"), tmp_path)
        ds = DicomLoader().load(path)
        assert ds.Modality in ("CR", "DX")

    @pytest.mark.sds("SDS-001")
    def test_wrong_modality_rejected(self, tmp_path):
        # TODO: build a mock with an unsupported modality (e.g. "CT"), save it with
        # save_mock(...), and assert that DicomLoader().load(...) rejects it.
        assert True  # <-- replace with your assertion

    # ── SDS-002: ViewClassifier accepts AP/PA, rejects everything else ──
    # classify() takes a dataset directly, so the mock goes straight in.
    @pytest.mark.sds("SDS-002")
    def test_ap_view_accepted(self):
        ds = make_dataset(view="AP")
        assert ViewClassifier().classify(ds) in ("AP", "PA")

    @pytest.mark.sds("SDS-002")
    def test_oblique_view_rejected(self):
        # TODO: build a mock with an unsupported view (e.g. "OBL") and assert that
        # ViewClassifier().classify(...) rejects it.
        assert True  # <-- replace with your assertion


### 📋 The unit test record

Running the test cell above writes `day2_data/release/unit_test_record.md` — a traceability
record that maps each test (by class and method) to the System Design Spec it verifies,
via its `@pytest.mark.sds(...)` marker, together with its latest pass/fail result. It lands
in `release/` alongside the other `make`-rendered QMS documents (described in the Requirements and Design notebook).

In [ ]:
# The traceability record built from the @pytest.mark.sds(...) markers in the cell above.
show_unit_test_record()

> ### 💡 Stuck? Load the worked tests
>
> Uncomment and then run the cell below **once** to pull the worked test suite into it (`%load` replaces the
> cell with the file contents and comments out the `%load` line). Then **run it again** to
> execute the tests.
>
> The suite is the completed version of the four tests above. The **two rejection tests
> will fail** — that is the point: they prove verification catches the two deliberate bugs.

In [ ]:
# %load src/test_handosteo.py

> ### 🟢 Optional bonus — make the tests pass (red → green)
>
> Your rejection tests fail because the stub has two deliberate bugs. Fix them and watch
> the tests go green — the core unit-testing workflow.
>
> The bug guards are commented out in `src/handosteo.py`.

# 6. Validation

Verification (step 5) asked *"did we build the software **right**?"*. Validation asks the complementary question: *"did we build the **right** software?"* — does it meet the user's real clinical need, not just the written specification?

Add a manual validation test to `day2_data/data/manual_tests.yml` that partially satisfies one of your requirements.

### Rules for manual test cases

- Must be executable by an end user (not necessarily the developer)
- Steps must be clearly written with a defined expected outcome
- Should map back to a system requirement (`SRS_XXX`)|

In [ ]:
# Display the contents of the manual validation tests yaml file.
show_yaml("day2_data/data/manual_tests.yml")

---
## 🎉 You've verified and validated HandOsteo

| Stage | What you did | Artefact |
|---|---|---|
| 5 · Verification | Unit tests for SDS-001 & SDS-002 | your `TestHandOsteo` / `src/test_handosteo.py` + `day2_data/release/unit_test_record.md` |
| 6 · Validation | Manual test cases | `manual_tests.yml` |

The `@pytest.mark.sds(...)` markers on your tests generated `unit_test_record.md`, the
traceability record linking each test back to the design spec it verifies.

Together with [`QMS_requirements_and_design.ipynb`](QMS_requirements_and_design.ipynb)
you've now walked the full traceable chain — requirement → hazard → design → test —
exactly how the [GSTT-CSC QMS-Template](https://github.com/GSTT-CSC/QMS-Template) turns
structured data into a release-ready QMS.